Tools In Langchain

1. How TO Create Tools
2. How to use built-in tolls and toolkits
3. How to use chat models to call tools
4. How to pass tool outputs to chat models

This @tool decorator is the simplest way to define a custom tool. The decorator uses the function name as the tool name by default, but this can be overridden by passing a string as the first argument. Additionally, the decorator will use the function's docstring as the tool's description - so a docstring MUST be provided.

In [1]:
## @tool decorter 
from langchain_core.tools import tool

@tool
async def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


In [2]:
from typing import Annotated,List

@tool
def multiply_by_max(
    a: Annotated[float, "First number"], 
    b: Annotated[List[float], "Second number"]
    ) -> float:
    """Multiply by the maximum b"""
    return a * max(b)

In [3]:
print(multiply_by_max.args)

{'a': {'description': 'First number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second number', 'items': {'type': 'number'}, 'title': 'B', 'type': 'array'}}


Structured Tool:
A Structured Tool is a tool that uses a defined input schema (structured inputs) instead of plain text.
👉 It tells the AI exactly:
•	what inputs are required 
•	their types (int, str, etc.) 
•	how to call the function correctly


In [4]:
from langchain_core.tools import StructuredTool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

async def amultiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


calculate = StructuredTool.from_function(func=multiply, coroutine=amultiply)

print(calculate.invoke({"a": 5, "b": 10}))
print(await calculate.ainvoke({"a": 5, "b": 5}))

50.0
25.0


INBUILD TOOLS

Wikipedia Integration


In [5]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
api_wrapper = WikipediaAPIWrapper(top_k_results=5,doc_content_chars_limit=500)
tool=WikipediaQueryRun(api_wrapper=api_wrapper)
print(tool.invoke({"query": "What is the capital of India?"}))


Page: Delhi
Summary: Delhi, officially the National Capital Territory (NCT) of Delhi, is a megacity and a union territory of India containing New Delhi, the capital of India. Straddling the Yamuna river, but spread chiefly to the west, or beyond its right bank, Delhi shares borders with the state of Uttar Pradesh in the east and with the state of Haryana in the remaining directions. Delhi became a union territory on 1 November 1956 and the NCT in 1995. The NCT covers an area of 1,484 square kilometres (573 sq mi). According to the 2011 census, Delhi's city proper population was over 11 million, while the NCT's population was about 16.8 million.
The topography of the medieval fort Purana Qila on the banks of the river Yamuna matches the literary description of the citadel Indraprastha in the Sanskrit epic Mahabharata; however, excavations in the area have revealed no signs of an ancient built environment.
From the early 13th century until the mid-19th century, Delhi was the capital of t

Call Tool With LLM Model

In [6]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=5,doc_content_chars_limit=100)
wiki_tool=WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query": "What is the capital of India?"}))


Page: Delhi
Summary: Delhi, officially the National Capital Territory (NCT) of Delhi, is a megacity and a union territory of India containing New Delhi, the capital of India. Straddling the Yamuna river, but spread chiefly to the west, or beyond its right bank, Delhi shares borders with the state of Uttar Pradesh in the east and with the state of Haryana in the remaining directions. Delhi became a union territory on 1 November 1956 and the NCT in 1995. The NCT covers an area of 1,484 square kilometres (573 sq mi). According to the 2011 census, Delhi's city proper population was over 11 million, while the NCT's population was about 16.8 million.
The topography of the medieval fort Purana Qila on the banks of the river Yamuna matches the literary description of the citadel Indraprastha in the Sanskrit epic Mahabharata; however, excavations in the area have revealed no signs of an ancient built environment.
From the early 13th century until the mid-19th century, Delhi was the capital of t

In [29]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

In [30]:
tools=[wiki_tool,add,multiply]

In [31]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Generative_Ai\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=5, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 StructuredTool(name='add', description='Add two numbers.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001DDE91E8CC0>),
 StructuredTool(name='multiply', description='Multiply two numbers.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001DDEAC74EA0>)]

In [32]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("qwen/qwen3-32b",model_provider="groq")
llm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001DDEAC456D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DDEAC460D0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [33]:
llm_with_tools=llm.bind_tools(tools)


In [37]:
from langchain_core.messages import HumanMessage
query="What is 2*3 and what is the capital of India?"
messages=[HumanMessage(query)]
response=llm_with_tools.invoke(query)
print(response)


content='' additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking two questions here: "What is 2*3" and "what is the capital of India?" First, I need to handle each part separately.\n\nFor the first part, 2 multiplied by 3. I remember there\'s a function called multiply that takes two numbers. So I should use that. The parameters are a and b, both numbers. So a=2 and b=3. That should give the product.\n\nThen the second part, the capital of India. That\'s a general knowledge question. The wikipedia function is useful here. The parameter is a query, so I\'ll input "capital of India" into that. Wikipedia should have the answer, which I think is New Delhi. But I need to confirm by calling the function.\n\nWait, do I need to make two separate tool calls? Yes, because there are two distinct questions. First, use multiply for the math part, then wikipedia for the capital. Let me structure the tool calls accordingly. Make sure each function is called with the correct ar

In [35]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'pf3ywj73n',
  'type': 'tool_call'},
 {'name': 'wikipedia',
  'args': {'query': 'capital of India'},
  'id': 'r1p7jvhbw',
  'type': 'tool_call'}]

In [38]:
for tool_call in response.tool_calls:
    selected_tool = {"add": add, "multiply": multiply, "wikipedia": wiki_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    
messages
    

[HumanMessage(content='What is 2*3 and what is the capital of India?', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6.0', name='multiply', tool_call_id='4teybpm3q'),
 ToolMessage(content='Page: List of capitals of India\nSummary: This is a list of locations which have served as capital cities in India. The current capital city is New Delhi, which replaced Calcutta in 1911.\n\n\n\nPage: National Capital Region (India)\nSummary: The National Capital Region (NCR; Rāṣṭrīya Rājadhānī Kṣetra) is a region centred on the city of Delhi, a special union territory of India that hosts the country\'s capital city New Delhi. It encompasses the entirety of Delhi and a number of adjacent districts from the states of Haryana, Uttar Pradesh, and Rajasthan. The NCR and the associated National Capital Region Planning Board (NCRPB) were created in 1985 to plan the development of the region and to evolve "harmonized policies for the control of land-uses and development of infrastructur

In [39]:
llm_with_tools.invoke(messages)

AIMessage(content='The answer to 2 multiplied by 3 is **6**.  \n\nThe capital of India is **New Delhi**, which is located within the National Capital Territory (NCT) of Delhi.', additional_kwargs={'reasoning_content': 'Okay, let me break down the user\'s question. They asked two things: what is 2*3 and what is the capital of India. \n\nFirst, the math part. 2 multiplied by 3 is straightforward. I remember that multiplication is just repeated addition, so 2 added three times is 6. The user probably expects a quick answer here. I should check if there\'s any trick or if they\'re testing something, but it seems simple. The function \'multiply\' is available, so I\'ll use that.\n\nNext, the capital of India. I know that India\'s capital is New Delhi. But wait, sometimes people might confuse Delhi with New Delhi. Let me make sure. New Delhi is the capital city, and it\'s located within the National Capital Territory of Delhi. The Wikipedia tool response mentions that Delhi is the National C